# 9 — Reproducing every figure in the note

Every figure, table and quoted number in the note is generated from the code
in this repository. Nothing is drawn by hand and nothing is typed in twice.
This notebook explains the machinery so you can rebuild any of it.

## Three stages, one rule

```
data/prepare.py      raw downloads  ->  data/processed/     (rarely run)
pipeline/run_*.py    processed + model  ->  results/        (the solving)
pipeline/build.py    results  ->  writing/generated/        (the drawing)
```

The rule that matters: **`build.py` reads only `results/`.** It imports no
model code and solves nothing, which is why redrawing a figure after a
wording change takes seconds instead of hours. If a figure needs a number
that is not in `results/`, the fix is to have the `run_*` stage write it out.

## Rebuilding a figure

From the note's directory:

```bash
python pipeline/run_dispatch.py          # solves section 2, writes results/
python pipeline/build.py                 # draws every figure it has data for
```

Figures land in `writing/generated/figures/`, and quoted numbers become LaTeX
macros in `writing/generated/tables/`, so the prose and the figures cannot
disagree.

## The instance-size parameter

Every `run_*.py` takes `--hours`. The default is small and solves instantly;
the value below reproduces the note's own figures.

| Note section | Command |
|---|---|
| §2 Dispatch | `python pipeline/run_dispatch.py` |
| §3 Intermittency | `python pipeline/run_dispatch_t.py --hours 8784` |
| §4 Storage | `python pipeline/run_storage.py --hours 8784` |
| §5 Heat | `python pipeline/run_heat.py --hours 8784` |
| §6 Transmission | `python pipeline/run_network.py --hours 1095` |
| §7 Investment | `python pipeline/run_greenfield.py --hours 1095` |
| §8 Expansion at European scale | `python pipeline/run_expansion.py --hours 1095` |
| §9 Uncertainty, weather | `python pipeline/run_weather.py --hours 1095` (after §7) |
| §9 Uncertainty, costs | `python pipeline/run_costs.py --hours 1095` (after §7) |
| §9 Uncertainty, tax vs cap | `python pipeline/run_taxcap.py --hours 1095` (after §8) |

Expensive solves are cached in `results/cache/`; delete the cache to force a
fresh solve.

In [ ]:
# Make the note's models importable, wherever you launched Jupyter from.
import sys
from pathlib import Path

here = Path.cwd()
note = next(p for p in [here, *here.parents] if (p / "model" / "dispatch.py").exists())
sys.path.insert(0, str(note))
PROCESSED = note / "data" / "processed"

import pandas as pd
import matplotlib.pyplot as plt

print(f"note root: {note}")

## Running a stage from inside the notebook

You do not have to leave Jupyter. The run scripts are ordinary modules. This
one solves the ten-generator dispatch of section 2 and writes its results;
a fresh clone has an empty `results/`, so run it before reading anything.


In [ ]:
import subprocess, sys
out = subprocess.run([sys.executable, str(note / "pipeline" / "run_dispatch.py")],
                     capture_output=True, text=True, cwd=str(note))
print(out.stdout[-800:] or out.stderr[-800:])

## What is in `results/` now

These are the files the figures are drawn from. They are ordinary CSV and
JSON — you can read them yourself.


In [ ]:
results = note / "results"
if results.exists():
    for f in sorted(results.glob("*")):
        if f.is_file():
            print(f"{f.name:42s} {f.stat().st_size/1024:8.1f} kB")
else:
    print("results/ is empty -- run a pipeline/run_*.py first")

In [ ]:
# Example: the marginal abatement cost curve of figure 2.4 comes from here.
sweep = pd.read_csv(results / "dispatch_cap_sweep.csv")
sweep.head()

## Your turn

1. Change the load in `pipeline/run_dispatch.py`, re-run it and `build.py`,
   and look at how figure 2.1 changes.
2. Open `results/dispatch_t_sweep.csv` and reproduce the cannibalisation
   figure yourself from the raw columns.
3. Find the line in `pipeline/build.py` that draws the marginal abatement
   cost curve, and change the assumed top-down curve's curvature.